In [0]:
from pyspark.sql import functions as F

BASE_PATH = "/Volumes/databricks_wrkspce/default/banking_data/migration"

SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"

In [0]:
GOLD_CONFIG = {

    "dimensions": {

        "dim_customer": {
            "source": "customers",
            "key": ["customer_id"],
            "columns": [
                "customer_id",
                "first_name",
                "last_name",
                "email",
                "city",
                "customer_segment",
                "date_of_birth"
            ]
        },

        "dim_account": {
            "source": "accounts",
            "key": ["account_id"],
            "columns": [
                "account_id",
                "customer_id",
                "branch_id",
                "account_open_date"
            ]
        },

        "dim_branch": {
            "source": "branches",
            "key": ["branch_id"],
            "columns": [
                "branch_id",
                "branch_name",
                "city",
                "state",
                "region"
            ]
        }
    },

    "facts": {

        "fact_transactions": {

            "source": "transactions",

            "joins": [
                {
                    "table": "accounts",
                    "left_key": "account_id",
                    "right_key": "account_id",
                    "join_type": "left"
                },

                {
                    "table": "customers",
                    "left_key": "customer_id",
                    "right_key": "customer_id",
                    "join_type": "left"
                },

                {
                    "table": "branches",
                    "left_key": "branch_id",
                    "right_key": "branch_id",
                    "join_type": "left"
                }
            ],

            "group_by": [
                "customer_id"
            ],

            "aggregations": {
                "transaction_count": {
                    "function": "count",
                    "column": "transaction_id"
                },

                "total_transaction_amount": {
                    "function": "sum",
                    "column": "amount"
                },

                "average_transaction_amount": {
                    "function": "avg",
                    "column": "amount"
                },

                "unique_transaction_types": {
                    "function": "countDistinct",
                    "column": "transaction_type"
                }
            }
        }
    }
}

In [0]:
def read_silver(table_name):

    return (
        spark.read
        .format("delta")
        .load(f"{SILVER_PATH}/{table_name}")
    )

In [0]:
def write_gold(df, table_name):

    # 1. Write Gold Delta data to Volume
    gold_path = f"{GOLD_PATH}/{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(gold_path)
    )

    # 2. Write the same data as a Unity Catalog table
    target_table = f"databricks_wrkspce.default.{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

In [0]:
def build_dimension(config):

    df = read_silver(config["source"])

    df = df.select(*config["columns"])

    if config.get("key"):
        df = df.dropDuplicates(config["key"])

    return df

In [0]:
def build_aggregation(df, group_by, aggregations):

    aggregation_expressions = []

    for output_column, rule in aggregations.items():

        function_name = rule["function"]
        source_column = rule["column"]

        if function_name == "count":
            expression = F.count(source_column)

        elif function_name == "sum":
            expression = F.sum(source_column)

        elif function_name == "avg":
            expression = F.avg(source_column)

        elif function_name == "countDistinct":
            expression = F.countDistinct(source_column)

        else:
            raise ValueError(
                f"Unsupported aggregation: {function_name}"
            )

        aggregation_expressions.append(
            expression.alias(output_column)
        )

    return (
        df.groupBy(*group_by)
        .agg(*aggregation_expressions)
    )

In [0]:
def apply_joins(df, joins, tables):

    for join_config in joins:

        table_name = join_config["table"]
        left_key = join_config["left_key"]
        right_key = join_config["right_key"]
        join_type = join_config.get("type", "left")

        right_df = tables[table_name]

        left_columns = df.columns

        right_columns = [
            c for c in right_df.columns
            if c != right_key and c not in left_columns
        ]

        df = (
            df.alias("left")
            .join(
                right_df.alias("right"),
                F.col(f"left.{left_key}") == F.col(f"right.{right_key}"),
                join_type
            )
            .select(
                *[
                    F.col(f"left.{c}").alias(c)
                    for c in left_columns
                ],
                *[
                    F.col(f"right.{c}").alias(c)
                    for c in right_columns
                ]
            )
        )

    return df

In [0]:
def load_silver_tables(config):

    tables = set()

    for dimension in config["dimensions"].values():
        tables.add(dimension["source"])

    for fact in config["facts"].values():

        tables.add(fact["source"])

        for join in fact["joins"]:
            tables.add(join["table"])

    return {
        table: read_silver(table)
        for table in tables
    }

In [0]:
silver_tables = load_silver_tables(GOLD_CONFIG)

for dimension_name, config in GOLD_CONFIG["dimensions"].items():

    print(f"Building {dimension_name}")

    dimension_df = build_dimension(config)

    write_gold(
        dimension_df,
        dimension_name
    )

Building dim_customer
Building dim_account
Building dim_branch


In [0]:
for fact_name, config in GOLD_CONFIG["facts"].items():

    print(f"\nBuilding {fact_name}")

    df = silver_tables[config["source"]]

    df = apply_joins(
        df,
        config["joins"],
        silver_tables
    )

    df = build_aggregation(
        df,
        config["group_by"],
        config["aggregations"]
    )

    write_gold(
        df,
        fact_name
    )


Building fact_transactions


In [0]:
gold_path = "/Volumes/databricks_wrkspce/default/banking_data/migration/gold"

display(dbutils.fs.ls(gold_path))

path,name,size,modificationTime
dbfs:/Volumes/databricks_wrkspce/default/banking_data/migration/gold/dim_account/,dim_account/,0,1787227465000
dbfs:/Volumes/databricks_wrkspce/default/banking_data/migration/gold/dim_branch/,dim_branch/,0,1787227467000
dbfs:/Volumes/databricks_wrkspce/default/banking_data/migration/gold/dim_customer/,dim_customer/,0,1787227463000
dbfs:/Volumes/databricks_wrkspce/default/banking_data/migration/gold/fact_transactions/,fact_transactions/,0,1787228705000
